In [ ]:
import cv2
import mediapipe as mp
import math
import numpy as np
import time
import os
#parameter changing due to bad/good posture
# Notify function for Mac
def notify_mac(title, message):
    os.system("afplay/System/Library/Sounds/Submarine.aiff")  
    os.system(f'''osascript -e 'display notification "{message}" with title "{title}"' ''')
def check(l1,img):
        dis=[]  
        for i in range (0,3):
            x1,y1=points_refference[i]
            x2,y2=l1[i]
            dis.append(np.sqrt((x1-x2)**2+(y1-y2)**2))
    
        
       
        head=int(dis[0]>distance*0.30)
        lefts=int(dis[1]>distance*0.25)
        rights=int(dis[2]>distance*0.25)
        trajectory_bad =False
        
        
        
        if(head+lefts+rights>1):
            trajectory_bad= True
        else:
            trajectory_bad = False  
        
        cv2.putText(frame, "Ratio : "+str("{:.2f}".format((dis[0]/distance)*100)),(980,190),
            cv2.FONT_HERSHEY_SIMPLEX,1,(0,0,255) if trajectory_bad else (0,255,0),2)
        cv2.putText(frame,"Left Ratio : "+str("{:.2f}".format((dis[1]/distance)*100)),(980,220),
            cv2.FONT_HERSHEY_SIMPLEX,1,(0,0,255) if trajectory_bad else (0,255,0),2)
        cv2.putText(frame,"Right ratio : "+str("{:.2f}".format((dis[2]/distance)*100)),(980,250),
            cv2.FONT_HERSHEY_SIMPLEX,1,(0,0,255) if trajectory_bad else (0,255,0),2)    
        
        return trajectory_bad 

# Setup
cap = cv2.VideoCapture(0)
mp_pose = mp.solutions.pose
mp_face = mp.solutions.face_mesh
pose = mp_pose.Pose(min_detection_confidence=0.5, min_tracking_confidence=0.5)
face_mesh = mp_face.FaceMesh()
baseline_right = None
baseline_left = None
baseline_nose_chest = None
bad_posture_start_time = None
notified = False

points_totrack=[0,11,12]
points_refference=[]
size=len(points_totrack)
distance=0

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    h, w, _ = frame.shape
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    pose_results = pose.process(rgb)
    face_results = face_mesh.process(rgb)
    frame = cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR)

    bad_posture = False
    y_diff = 0

    if pose_results.pose_landmarks:
        l1=[]
        for i in points_totrack:
            x=int(pose_results.pose_landmarks.landmark[i].x*1280)
            y=int(pose_results.pose_landmarks.landmark[i].y*720)
            l1.append((x,y))
            cv2.circle(frame,(x,y),4,(255,0,0),-1)
        lm = pose_results.pose_landmarks.landmark

        # Shoulder coordinates
        r_sh = lm[12]
        l_sh = lm[11]
        x1, y1 = int(r_sh.x * w), int(r_sh.y * h)
        x2, y2 = int(l_sh.x * w), int(l_sh.y * h)
        mid_x, mid_y = (x1 + x2) // 2, (y1 + y2) // 2

        # Nose and chest (midpoint of shoulders)
        nose = lm[0]
        nose_x, nose_y = int(nose.x * w), int(nose.y * h)
        chest_x, chest_y = int((r_sh.x + l_sh.x) / 2 * w), int((r_sh.y + l_sh.y) / 2 * h)

        # Draw points and lines
        cv2.circle(frame, (x1, y1),5, (255, 0, 0),-1 )
        cv2.circle(frame, (x2, y2), 5, (0, 255, 0), -1)
        #cv2.circle(frame, (mid_x, mid_y), 5, (0, 0, 255), -1)
        cv2.circle(frame, (nose_x, nose_y), 5, (255, 0, 0), -1)
        cv2.circle(frame, (chest_x, chest_y), 5, (0, 255, 0), -1)

        cv2.line(frame, (mid_x, mid_y), (x1, y1), (0, 255, 255), 2)
        cv2.line(frame, (mid_x, mid_y), (x2, y2), (0, 255, 255), 2)
        cv2.line(frame, (nose_x, nose_y), (chest_x, chest_y), (0, 255, 255), 2)

        # Distance calculations
        def dist(xa, ya, xb, yb):
            return int(math.sqrt((xa - xb) ** 2 + (ya - yb) ** 2))

        d_mid_right = dist(mid_x, mid_y, x1, y1)
        d_mid_left = dist(mid_x, mid_y, x2, y2)
        nose_chest_dist = dist(nose_x, nose_y, chest_x, chest_y)

        # Display distances
        cv2.putText(frame, f"{d_mid_right}px", ((mid_x + x1)//2 + 10, (mid_y + y1)//2),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 0), 2)
        cv2.putText(frame, f"{d_mid_left}px", ((mid_x + x2)//2 + 10, (mid_y + y2)//2),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 0), 2)
        cv2.putText(frame, f"{nose_chest_dist}px", ((nose_x + chest_x)//2 + 10, (nose_y + chest_y)//2),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 0), 2)

        # Head tilt detection
        if face_results.multi_face_landmarks:
            for face_landmarks in face_results.multi_face_landmarks:
                left_eye = face_landmarks.landmark[33]
                right_eye = face_landmarks.landmark[263]
                left_eye_y = int(left_eye.y * h)
                right_eye_y = int(right_eye.y * h)
                y_diff = abs(left_eye_y - right_eye_y)

                cv2.circle(frame, (int(left_eye.x * w), left_eye_y), 5, (255, 0, 0), -1)
                cv2.circle(frame, (int(right_eye.x * w), right_eye_y), 5, (0, 255, 0), -1)

        # Initialize baselines
    key_input=cv2.waitKey(1) & 0xFF
    if(key_input==ord('r')):
        if(l1 != []):
            points_refference=l1
            distance=(l1[0][0]-((l1[1][0]+l1[2][0]))/2)**2
            distance+=(l1[0][1]-((l1[1][1]+l1[2][1]))/2)**2
            distance=np.sqrt(distance)
            baseline_right = d_mid_right
            baseline_left = d_mid_left
            baseline_nose_chest = nose_chest_dist
            print(points_refference)
            print(distance)
            print("Setting done")


    if points_refference==[]:
        cv2.putText(frame,"REFERENCE NOT SET",(430,40),cv2.FONT_HERSHEY_SIMPLEX,1.25,(255,0,0),2)
    else:
        # Posture checks
        change_right = abs(d_mid_right - baseline_right) / baseline_right * 100
        change_left = abs(d_mid_left - baseline_left) / baseline_left * 100
        change_nose = abs(nose_chest_dist - baseline_nose_chest) / baseline_nose_chest * 100
        
        trajectory=check(l1,frame)
        nc_bad=False
        lsc_bad=False
        if change_right > 10 or change_left > 10:
            bad_posture = True
            lsc_bad=True
        if (change_nose > 90 and nose_chest_dist < baseline_nose_chest) or (nose_y > chest_y):
            bad_posture = True
            nc_bad=True
        
        if y_diff > 60:
            bad_posture = True
        elif trajectory:
             bad_posture= True
             
        cv2.putText(frame, f"NCD : {change_nose:.2f}", (980, 100),
            cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255) if nc_bad else (0, 255, 0), 2)
        cv2.putText(frame, f"LSCD : {change_left:.2f}", (980, 130),
                    cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255) if lsc_bad else (0, 255, 0), 2)
        cv2.putText(frame, f"RSCD : {change_right:.2f}", (980, 160),
                    cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255) if lsc_bad else (0, 255, 0), 2)
    # Posture notification logic
        if bad_posture:
            if bad_posture_start_time is None:
                bad_posture_start_time = time.time()
            elif time.time() - bad_posture_start_time >= 5 and not notified:
                notify_mac("Posture Alert", "Please sit straight!")
                notified = True

            if int(time.time() * 2) % 2 == 0:
                cv2.putText(frame, "** BAD POSTURE **", (875, 50),
                            cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 0, 255), 3)
        else:
            bad_posture_start_time = None
            notified = False
            cv2.putText(frame, "Good Posture", (1000, 50),
                        cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 255, 0), 2)
    
    
        


    cv2.imshow("Posture Monitor", frame)
    if key_input == ord('q'):
        break


cap.release()
cv2.destroyAllWindows()
